# 15 — LASSO Feature Selection

Screens the country-year feature panel from Notebook 14 using L1-regularised logistic
regression (LASSO), then rescues non-linearly associated features that LASSO would
silently discard.

## Method

1. **LASSO screening** — `LogisticRegressionCV(penalty='l1', solver='saga')` with 5
   expanding temporal folds inside the training window (2000–2018). λ is chosen by
   the 1-SE rule (most regularised λ within 1 standard error of best AUPRC).
2. **Mutual information rescue** — `mutual_info_classif` detects features with
   U-shaped, threshold, or interaction-only effects that receive a zero LASSO
   coefficient despite having real predictive information. Features in the MI top-50
   that were zeroed by LASSO and show a large MI-vs-LASSO rank gap are rescued.
3. **Audit table** — per feature: LASSO coefficient, MI score, LASSO rank, MI rank,
   rank gap, rescued flag. Written to ADLS for use in Notebook 17.

## Outputs per outcome
- `feature_selection/{RUN_DATE}/selected_{outcome}.json` — feature manifest
- `feature_selection/{RUN_DATE}/audit_{outcome}.parquet` — audit table
- MLflow: regularisation path plot (λ vs. coefficient magnitude)

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER       (default: 'data')
AZUREML_MLFLOW_URI   (optional, for MLflow tracking)
```

In [ ]:
import os
import json
import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import rankdata

from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import BaseCrossValidator

from azure.identity import DefaultAzureCredential
import adlfs
import mlflow

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

TRAIN_END_YEAR    = 2018   # must match notebook 14

OUTCOMES = [
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
]

# Mutual information rescue thresholds
MI_TOP_N       = 50   # feature must be in MI top-N to be rescue-eligible
MI_RANK_GAP    = 30   # MI rank must exceed LASSO rank by at least this much

LASSO_N_CS     = 30   # number of regularisation strengths to try
LASSO_MAX_ITER = 5000
RANDOM_STATE   = 42

print(f"Run date       : {RUN_DATE}")
print(f"Train window   : ≤{TRAIN_END_YEAR}")
print(f"Outcomes       : {OUTCOMES}")

## ADLS helpers

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str) -> pd.DataFrame | None:
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: prefix not found: {full_prefix}")
        return None
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions found under {full_prefix}")
        return None
    latest_dir = date_dirs[0]
    parquet_files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(f"{latest_dir}/*.parquet")
    ]
    if not parquet_files:
        print(f"  WARNING: no .parquet files in {latest_dir}")
        return None
    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in parquet_files]
    df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows from {latest_dir}")
    return df

def write_json(obj: dict, subpath: str) -> None:
    import io
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    path = f"{ADLS_CONTAINER}/{subpath}"
    with fs.open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"  Written JSON → {path}")

## Load feature matrix and labels

In [ ]:
df_features = read_latest_parquet("processed/feature_matrix")
df_labels   = read_latest_parquet("processed/feature_matrix")  # same partition

# labels.parquet is a separate file in the same directory — reload it explicitly
if df_features is not None:
    # Re-read labels separately (read_latest_parquet concatenates all parquets
    # in the partition; we need to separate features from labels)
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    prefix = f"{ADLS_CONTAINER}/processed/feature_matrix"
    entries = fs.ls(prefix, detail=False)
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)], reverse=True
    )
    if date_dirs:
        latest = date_dirs[0]
        def _read_one(filename):
            files = fs.glob(f"{latest}/{filename}")
            if not files:
                return None
            path = (
                f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
                + files[0].replace(f"{ADLS_CONTAINER}/", "", 1)
            )
            return pd.read_parquet(path, storage_options=storage_options)

        df_features = _read_one("feature_matrix.parquet")
        df_labels   = _read_one("labels.parquet")

OUTCOME_COLS = [
    "civil_war_onset", "coup_attempt", "regime_backsliding",
    "mass_unrest_onset", "humanitarian_crisis_onset",
]
ID_COLS = ["iso3", "year"]

if df_features is not None and df_labels is not None:
    feat_cols = [c for c in df_features.columns if c not in ID_COLS + OUTCOME_COLS]
    print(f"Feature matrix : {df_features.shape}")
    print(f"Labels         : {df_labels.shape}")
    print(f"Feature columns: {len(feat_cols)}")
else:
    print("ERROR: could not load feature matrix or labels from ADLS")

## Temporal cross-validation splitter

Five expanding-window folds, all within the training period (2000–2018).
No shuffling across time — each fold's validation set is strictly later than its train set.

In [ ]:
class ExpandingYearCV(BaseCrossValidator):
    """
    Expanding-window temporal CV on a country-year panel.

    The training window grows year by year; each fold's validation set is the
    next `val_years` years. Only folds where the validation set has at least
    `min_val_positives` positive examples are kept.
    """

    def __init__(self, years: np.ndarray, n_splits: int = 5, val_years: int = 2,
                 min_val_positives: int = 2):
        self.years           = years
        self.n_splits        = n_splits
        self.val_years       = val_years
        self.min_val_positives = min_val_positives

    def split(self, X, y=None, groups=None):
        unique_years = np.sort(np.unique(self.years))
        # Candidate cutpoints: all but the last val_years years
        candidates = unique_years[: -self.val_years]
        # Evenly space n_splits cutpoints across candidates
        step = max(1, len(candidates) // self.n_splits)
        cutpoints = candidates[step - 1 :: step][: self.n_splits]

        for cutpoint in cutpoints:
            train_idx = np.where(self.years <= cutpoint)[0]
            val_idx   = np.where(
                (self.years > cutpoint) &
                (self.years <= cutpoint + self.val_years)
            )[0]
            if y is not None and len(val_idx) > 0:
                n_pos = np.sum(np.array(y)[val_idx] == 1)
                if n_pos < self.min_val_positives:
                    continue
            if len(train_idx) and len(val_idx):
                yield train_idx, val_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

print("ExpandingYearCV defined")